# Práctica 4. Procesamiento audio
## Ejercicio 1
Construir un identificador de notas musicales. Es decir; en su versión más sencilla  (y  suficiente) la entrada es un sonido con una sola nota musical y debe identificar cuál es. Por simplicidad  elija un único instrumento para la identificación.  


In [39]:
import numpy as np
from scipy.fft import fft, fftfreq
from scipy.signal import find_peaks
from scipy.io import wavfile

NOTES_SPANISH = {
    'C': 'Do', 'C#': 'Do#', 'Db': 'Reb', 'D': 'Re', 'D#': 'Re#', 'Eb': 'Mib',
    'E': 'Mi', 'F': 'Fa', 'F#': 'Fa#', 'Gb': 'Solb', 'G': 'Sol',
    'G#': 'Sol#', 'Ab': 'Lab', 'A': 'La', 'A#': 'La#', 'Bb': 'Sib', 'B': 'Si'
}

EQUIVALENCIAS = {
    'Do': 'Do', 'Do#': 'Do#/Reb', 'Re': 'Re', 'Re#': 'Re#/Mib',
    'Mi': 'Mi', 'Fa': 'Fa', 'Fa#': 'Fa#/Solb', 'Sol': 'Sol',
    'Sol#': 'Sol#/Lab', 'La': 'La', 'La#': 'La#/Sib', 'Si': 'Si'
}

def identificar_nota(archivo_audio):
    sample_rate, audio_data = wavfile.read(archivo_audio)
    
    if len(audio_data.shape) == 2:
        audio_data = audio_data.mean(axis=1)
    
    audio_data = audio_data.astype(float)
    
    N = len(audio_data)
    yf = fft(audio_data)
    xf = fftfreq(N, 1/sample_rate)
    
    yf_abs = np.abs(yf[:N//2])
    xf_positive = xf[:N//2]
    
    peaks, _ = find_peaks(yf_abs, height=np.max(yf_abs)*0.1)
    
    if len(peaks) > 0:
        fundamental_idx = peaks[np.argmax(yf_abs[peaks])]
        frecuencia_fundamental = abs(xf_positive[fundamental_idx])
    else:
        fundamental_idx = np.argmax(yf_abs)
        frecuencia_fundamental = abs(xf_positive[fundamental_idx])
    
    notas = ['Do', 'Do#', 'Re', 'Re#', 'Mi', 'Fa', 'Fa#', 'Sol', 'Sol#', 'La', 'La#', 'Si']
    
    frecuencias_base = {
        'Do': 16.35, 'Do#': 17.32, 'Re': 18.35, 'Re#': 19.45,
        'Mi': 20.60, 'Fa': 21.83, 'Fa#': 23.12, 'Sol': 24.50,
        'Sol#': 25.96, 'La': 27.50, 'La#': 29.14, 'Si': 30.87
    }
    
    min_diferencia = float('inf')
    nota_detectada = None
    
    for nota in notas:
        freq_base = frecuencias_base[nota]
        freq_nota = freq_base
        
        while freq_nota < 8000:
            diferencia = abs(frecuencia_fundamental - freq_nota)
            if diferencia < min_diferencia:
                min_diferencia = diferencia
                nota_detectada = nota
            freq_nota *= 2
    
    return nota_detectada, frecuencia_fundamental

if __name__ == "__main__":
    for nota_original in ["A", "Ab", "B", "Bb", "C", "D", "Db", "E", "Eb", "F", "G", "Gb"]:
        archivo = f"data/E1/piano/Piano.ff.{nota_original}4.wav"
        nota, frecuencia = identificar_nota(archivo)
        print(f"Nota original : {NOTES_SPANISH[nota_original]}")
        if nota in EQUIVALENCIAS:
            print(f"Nota detectada: {EQUIVALENCIAS[nota]}")
        else:
            print(f"Nota detectada: {nota}")
        print(f"Frecuencia fundamental: {frecuencia:.2f} Hz")
        print()


Nota original : La
Nota detectada: La
Frecuencia fundamental: 440.79 Hz

Nota original : Lab
Nota detectada: Sol#/Lab
Frecuencia fundamental: 416.91 Hz

Nota original : Si
Nota detectada: Si
Frecuencia fundamental: 494.76 Hz

Nota original : Sib
Nota detectada: La#/Sib
Frecuencia fundamental: 467.03 Hz

Nota original : Do
Nota detectada: Do
Frecuencia fundamental: 262.17 Hz

Nota original : Re
Nota detectada: Re
Frecuencia fundamental: 588.99 Hz

Nota original : Reb
Nota detectada: Do#/Reb
Frecuencia fundamental: 556.36 Hz

Nota original : Mi
Nota detectada: Mi
Frecuencia fundamental: 330.87 Hz

Nota original : Mib
Nota detectada: Re#/Mib
Frecuencia fundamental: 624.19 Hz

Nota original : Fa
Nota detectada: Fa
Frecuencia fundamental: 350.37 Hz

Nota original : Sol
Nota detectada: Sol
Frecuencia fundamental: 392.76 Hz

Nota original : Solb
Nota detectada: Fa#/Solb
Frecuencia fundamental: 370.49 Hz




(APORTES ADICIONALES) 

a) El caso más sencillo es el del piano, pero se valorará que se haga con otros instrumentos como la guitarra, la trompeta...  

b) Se valorará que se identifiquen octavas de notas 

c) Identificación de acordes (complejo pero espectacular) 

d) Aportes adicionales

## Prueba Detector de octavas
Lo hace bien de 3-7

In [40]:
import numpy as np
from scipy.fft import fft, fftfreq
from scipy.signal import find_peaks
from scipy.io import wavfile

NOTES_SPANISH = {
    'C': 'Do', 'C#': 'Do#', 'Db': 'Reb', 'D': 'Re', 'D#': 'Re#', 'Eb': 'Mib',
    'E': 'Mi', 'F': 'Fa', 'F#': 'Fa#', 'Gb': 'Solb', 'G': 'Sol',
    'G#': 'Sol#', 'Ab': 'Lab', 'A': 'La', 'A#': 'La#', 'Bb': 'Sib', 'B': 'Si'
}

EQUIVALENCIAS = {
    'Do': 'Do', 'Do#': 'Do#/Reb', 'Re': 'Re', 'Re#': 'Re#/Mib',
    'Mi': 'Mi', 'Fa': 'Fa', 'Fa#': 'Fa#/Solb', 'Sol': 'Sol',
    'Sol#': 'Sol#/Lab', 'La': 'La', 'La#': 'La#/Sib', 'Si': 'Si'
}

def identificar_nota(archivo_audio):
    sample_rate, audio_data = wavfile.read(archivo_audio)
    
    if len(audio_data.shape) == 2:
        audio_data = audio_data.mean(axis=1)
    
    audio_data = audio_data.astype(float)
    
    window = np.hamming(len(audio_data))
    audio_windowed = audio_data * window
    
    N = len(audio_windowed)
    yf = fft(audio_windowed)
    xf = fftfreq(N, 1/sample_rate)
    
    yf_abs = np.abs(yf[:N//2])
    xf_positive = xf[:N//2]
    
    mask = (xf_positive > 50) & (xf_positive < 5000)
    yf_filtered = yf_abs.copy()
    yf_filtered[~mask] = 0
    
    peaks, properties = find_peaks(yf_filtered, height=np.max(yf_filtered)*0.2, distance=10)
    
    if len(peaks) > 0:
        peak_frequencies = xf_positive[peaks]
        peak_amplitudes = yf_abs[peaks]
        
        sorted_indices = np.argsort(peak_amplitudes)[::-1]
        
        best_freq = None
        max_score = -1
        
        for idx in sorted_indices[:10]:
            freq = peak_frequencies[idx]
            amp = peak_amplitudes[idx]
            
            harmonic_support = 0
            for h in range(2, 5):
                harmonic_freq = freq * h
                tolerance = freq * 0.03
                harmonic_peak = np.any((peak_frequencies > harmonic_freq - tolerance) & 
                                      (peak_frequencies < harmonic_freq + tolerance))
                if harmonic_peak:
                    harmonic_support += 1
            
            score = amp * (1 + harmonic_support * 0.5)
            
            if score > max_score:
                max_score = score
                best_freq = freq
        
        frecuencia_fundamental = best_freq
    else:
        fundamental_idx = np.argmax(yf_filtered)
        frecuencia_fundamental = abs(xf_positive[fundamental_idx])
    
    notas = ['Do', 'Do#', 'Re', 'Re#', 'Mi', 'Fa', 'Fa#', 'Sol', 'Sol#', 'La', 'La#', 'Si']
    
    frecuencias_base = {
        'Do': 16.3516, 'Do#': 17.3239, 'Re': 18.3540, 'Re#': 19.4454,
        'Mi': 20.6017, 'Fa': 21.8268, 'Fa#': 23.1246, 'Sol': 24.4997,
        'Sol#': 25.9565, 'La': 27.5000, 'La#': 29.1353, 'Si': 30.8677
    }
    
    min_diferencia = float('inf')
    nota_detectada = None
    octava_detectada = None

    for nota in notas:
        freq_base = frecuencias_base[nota]
        
        for octava in range(0, 10):
            freq_nota = freq_base * (2 ** octava)
            diferencia = abs(frecuencia_fundamental - freq_nota)
            
            if diferencia < min_diferencia:
                min_diferencia = diferencia
                nota_detectada = nota
                octava_detectada = octava

    return nota_detectada, octava_detectada, frecuencia_fundamental


if __name__ == "__main__":
    octavas = [0, 1, 2, 3, 4, 5, 6, 7, 8]
    
    for octava_original in octavas:
        print(f"=== OCTAVA {octava_original} ===")
        for nota_original in ["A", "Ab", "B", "Bb", "C", "D", "Db", "E", "Eb", "F", "G", "Gb"]:
            archivo = f"data/E1/piano/Piano.ff.{nota_original}{octava_original}.wav"
            try:
                nota, octava, frecuencia = identificar_nota(archivo)
                print(f"Nota original : {NOTES_SPANISH[nota_original]} (Octava {octava_original})")
                if nota in EQUIVALENCIAS:
                    print(f"Nota detectada: {EQUIVALENCIAS[nota]} (Octava {octava})")
                else:
                    print(f"Nota detectada: {nota} (Octava {octava})")
                print(f"Frecuencia fundamental: {frecuencia:.2f} Hz")
                print()
            except FileNotFoundError:
                pass
        print()

=== OCTAVA 0 ===
Nota original : La (Octava 0)
Nota detectada: Fa (Octava 2)
Frecuencia fundamental: 86.34 Hz

Nota original : Si (Octava 0)
Nota detectada: Sol (Octava 3)
Frecuencia fundamental: 195.35 Hz

Nota original : Sib (Octava 0)
Nota detectada: Sol (Octava 3)
Frecuencia fundamental: 195.35 Hz


=== OCTAVA 1 ===
Nota original : La (Octava 1)
Nota detectada: Mi (Octava 3)
Frecuencia fundamental: 164.70 Hz

Nota original : Lab (Octava 1)
Nota detectada: Sol#/Lab (Octava 2)
Frecuencia fundamental: 103.42 Hz

Nota original : Do (Octava 1)
Nota detectada: Sol (Octava 3)
Frecuencia fundamental: 195.37 Hz

Nota original : Mi (Octava 1)
Nota detectada: Sol#/Lab (Octava 3)
Frecuencia fundamental: 204.84 Hz

Nota original : Mib (Octava 1)
Nota detectada: Sol (Octava 3)
Frecuencia fundamental: 194.22 Hz

Nota original : Fa (Octava 1)
Nota detectada: Fa (Octava 2)
Frecuencia fundamental: 86.69 Hz

Nota original : Sol (Octava 1)
Nota detectada: Sol (Octava 2)
Frecuencia fundamental: 97.68 H

### Guitarra


In [55]:
import numpy as np
from scipy.fft import fft, fftfreq
from scipy.signal import find_peaks
from scipy.io import wavfile
from pathlib import Path
import re

NOTES_SPANISH = {
    'C': 'Do', 'C#': 'Do#', 'Db': 'Reb', 'D': 'Re', 'D#': 'Re#', 'Eb': 'Mib',
    'E': 'Mi', 'F': 'Fa', 'F#': 'Fa#', 'Gb': 'Solb', 'G': 'Sol',
    'G#': 'Sol#', 'Ab': 'Lab', 'A': 'La', 'A#': 'La#', 'Bb': 'Sib', 'B': 'Si'
}

EQUIVALENCIAS = {
    'Do': 'Do', 'Do#': 'Do#/Reb', 'Re': 'Re', 'Re#': 'Re#/Mib',
    'Mi': 'Mi', 'Fa': 'Fa', 'Fa#': 'Fa#/Solb', 'Sol': 'Sol',
    'Sol#': 'Sol#/Lab', 'La': 'La', 'La#': 'La#/Sib', 'Si': 'Si'
}

NOTE2SEMITONE = {
    "C":0, "C#":1, "Db":1, "D":2, "D#":3, "Eb":3, "E":4,
    "F":5, "F#":6, "Gb":6, "G":7, "G#":8, "Ab":8, "A":9, "A#":10, "Bb":10, "B":11
}
SEMITONE2NOTE_SHARP = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]

def note_to_midi(note_str: str) -> int:
    m = re.match(r"^([A-Ga-g])([b#]?)(-?\d+)$", note_str.strip())
    if not m:
        raise ValueError(f"Nota inválida: {note_str}")
    l, acc, octv = m.groups()
    l = l.upper(); acc = acc if acc else ""
    semitone = NOTE2SEMITONE[l + acc]
    octv = int(octv)
    return (octv + 1) * 12 + semitone

def midi_to_note(m: int) -> str:
    name = SEMITONE2NOTE_SHARP[m % 12]
    octv = m // 12 - 1
    return f"{name}{octv}"

def expand_range(start_note: str, end_note: str):
    m0 = note_to_midi(start_note)
    m1 = note_to_midi(end_note)
    step = 1 if m1 >= m0 else -1
    return [midi_to_note(m) for m in range(m0, m1 + step, step)]

def detectar_energia(audio_data, sample_rate, num_notas_esperadas):
    duracion_total = len(audio_data) / sample_rate
    duracion_por_nota = duracion_total / num_notas_esperadas
    samples_por_nota = int(duracion_por_nota * sample_rate)
    
    segmentos = []
    for i in range(num_notas_esperadas):
        start = i * samples_por_nota
        end = min((i + 1) * samples_por_nota, len(audio_data))
        
        if end - start > sample_rate * 0.05:
            segmentos.append((start, end))
    
    return segmentos

def identificar_nota_segmento(audio_chunk, sample_rate):
    if len(audio_chunk) < 2048:
        return None, None, None
    
    skip_attack = int(len(audio_chunk) * 0.1)
    audio_stable = audio_chunk[skip_attack:]
    
    if len(audio_stable) < 1024:
        audio_stable = audio_chunk
    
    window = np.hamming(len(audio_stable))
    audio_windowed = audio_stable * window
    
    N = len(audio_windowed)
    yf = fft(audio_windowed)
    xf = fftfreq(N, 1/sample_rate)
    
    yf_abs = np.abs(yf[:N//2])
    xf_positive = xf[:N//2]
    
    mask = (xf_positive > 70) & (xf_positive < 2000)
    yf_filtered = yf_abs.copy()
    yf_filtered[~mask] = 0
    
    peaks, _ = find_peaks(yf_filtered, height=np.max(yf_filtered)*0.15, distance=5)
    
    if len(peaks) == 0:
        return None, None, None
    
    peak_frequencies = xf_positive[peaks]
    peak_amplitudes = yf_abs[peaks]
    
    sorted_indices = np.argsort(peak_amplitudes)[::-1]
    
    best_freq = None
    max_score = -1
    
    for idx in sorted_indices[:8]:
        freq = peak_frequencies[idx]
        amp = peak_amplitudes[idx]
        
        harmonic_support = 0
        for h in range(2, 6):
            harmonic_freq = freq * h
            tolerance = freq * 0.04
            harmonic_peak = np.any((peak_frequencies > harmonic_freq - tolerance) & 
                                  (peak_frequencies < harmonic_freq + tolerance))
            if harmonic_peak:
                harmonic_support += 1
        
        score = amp * (1 + harmonic_support * 0.4)
        
        if score > max_score:
            max_score = score
            best_freq = freq
    
    if best_freq is None:
        return None, None, None
    
    frecuencia_fundamental = best_freq
    
    notas = ['Do', 'Do#', 'Re', 'Re#', 'Mi', 'Fa', 'Fa#', 'Sol', 'Sol#', 'La', 'La#', 'Si']
    
    frecuencias_base = {
        'Do': 16.3516, 'Do#': 17.3239, 'Re': 18.3540, 'Re#': 19.4454,
        'Mi': 20.6017, 'Fa': 21.8268, 'Fa#': 23.1246, 'Sol': 24.4997,
        'Sol#': 25.9565, 'La': 27.5000, 'La#': 29.1353, 'Si': 30.8677
    }
    
    min_diferencia = float('inf')
    nota_detectada = None
    octava_detectada = None

    for nota in notas:
        freq_base = frecuencias_base[nota]
        
        for octava in range(0, 10):
            freq_nota = freq_base * (2 ** octava)
            diferencia = abs(frecuencia_fundamental - freq_nota)
            
            if diferencia < min_diferencia:
                min_diferencia = diferencia
                nota_detectada = nota
                octava_detectada = octava

    return nota_detectada, octava_detectada, frecuencia_fundamental

def identificar_notas_guitarra(archivo_audio, notas_esperadas):
    sample_rate, audio_data = wavfile.read(archivo_audio)
    
    if len(audio_data.shape) == 2:
        audio_data = audio_data.mean(axis=1)
    
    audio_data = audio_data.astype(float)
    
    segmentos = detectar_energia(audio_data, sample_rate, len(notas_esperadas))
    
    if len(segmentos) == 0:
        return []
    
    resultados = []
    for i, (start, end) in enumerate(segmentos):
        if i >= len(notas_esperadas):
            break
        
        audio_chunk = audio_data[start:end]
        nota, octava, freq = identificar_nota_segmento(audio_chunk, sample_rate)
        
        if nota:
            resultados.append({
                'esperada': notas_esperadas[i],
                'detectada': nota,
                'octava': octava,
                'frecuencia': freq
            })
    
    return resultados

if __name__ == "__main__":
    base_dir = Path("data/E1/guitar")
    archivos = sorted(base_dir.glob("Guitar.*.wav"))
    
    NAME_REGEX_GTR = re.compile(
        r"^Guitar\.(?P<dynamic>pp|mf|ff)\.sul_?(?P<string>[EADGB])\."
        r"(?P<start>[A-G](?:[b#])?-?\d+)"
        r"(?P<end>[A-G](?:[b#])?-?\d+)\."
        r"(?P<channel>mono|stereo)\.wav$",
        re.IGNORECASE
    )
    
    for archivo in archivos:
        m = NAME_REGEX_GTR.match(archivo.name)
        if not m:
            continue
        
        info = m.groupdict()
        def norm(n): return n[0].upper() + (n[1:].lower() if len(n) > 1 else "")
        info["start"] = norm(info["start"])
        info["end"] = norm(info["end"])
        
        notas_esperadas = expand_range(info["start"], info["end"])
        
        print(f"\n{'='*60}")
        print(f"Archivo: {archivo.name}")
        print(f"Cuerda: {info['string']} | Dinámica: {info['dynamic']}")
        print(f"Notas esperadas: {' → '.join(notas_esperadas)}")
        print(f"{'='*60}")
        
        resultados = identificar_notas_guitarra(str(archivo), notas_esperadas)
        
        for r in resultados:
            nota_esp = r['esperada']
            mm = re.match(r"^([A-G]#?)(-?\d+)$", nota_esp)
            if mm:
                base_note, octv = mm.groups()
                nota_esp_spanish = NOTES_SPANISH.get(base_note, base_note) + octv
            else:
                nota_esp_spanish = nota_esp
            
            nota_det = EQUIVALENCIAS.get(r['detectada'], r['detectada'])
            print(f"Esperada: {nota_esp_spanish:8} | Detectada: {nota_det} (Octava {r['octava']}) | Freq: {r['frecuencia']:.2f} Hz")


Archivo: Guitar.ff.sulA.A2B2.stereo.wav
Cuerda: A | Dinámica: ff
Notas esperadas: A2 → A#2 → B2
Esperada: La2      | Detectada: La (Octava 2) | Freq: 108.06 Hz
Esperada: La#2     | Detectada: La#/Sib (Octava 2) | Freq: 115.03 Hz
Esperada: Si2      | Detectada: Si (Octava 2) | Freq: 121.92 Hz

Archivo: Guitar.ff.sulA.C3B3.stereo.wav
Cuerda: A | Dinámica: ff
Notas esperadas: C3 → C#3 → D3 → D#3 → E3 → F3 → F#3 → G3 → G#3 → A3 → A#3 → B3
Esperada: Do3      | Detectada: Do (Octava 3) | Freq: 128.97 Hz
Esperada: Do#3     | Detectada: Do#/Reb (Octava 4) | Freq: 274.69 Hz
Esperada: Re3      | Detectada: Re (Octava 4) | Freq: 289.19 Hz
Esperada: Re#3     | Detectada: Re#/Mib (Octava 3) | Freq: 153.48 Hz
Esperada: Mi3      | Detectada: Mi (Octava 3) | Freq: 163.08 Hz
Esperada: Fa3      | Detectada: Mi (Octava 3) | Freq: 162.77 Hz
Esperada: Fa#3     | Detectada: Fa#/Solb (Octava 3) | Freq: 181.46 Hz
Esperada: Sol3     | Detectada: Sol (Octava 3) | Freq: 192.18 Hz
Esperada: Sol#3    | Detectada: